In [1]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu126

!apt-get update
!apt-get install -y sox libsox-dev libsox-fmt-all

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 206.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 237.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 140.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 76.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 108.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 163.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Cell 1: Environment Setup
!pip install transformers accelerate jiwer --no-deps --quietimport os
import torch
import numpy as np
import pandas as pd
import soundfile as sf
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from sklearn.model_selection import train_test_split

# Thiết lập thiết bị và cố định seed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)
print(f"Using device: {device}")


Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: --quietimport
Using device: cuda


In [ ]:
#CELL 2
import os
os.environ["HF_HOME"] = "/kaggle/working/hf_cache_mdd_v2"

class Config:
    # --- ĐƯỜNG DẪN TẬP HUẤN LUYỆN (TRAIN) ---
    TRAIN_CSV = "/kaggle/input/datasets/trngquangthi4139/mdd-challenge/MDD-Challenge-2025-training-set/metadata/train_phones.csv" 
    TRAIN_AUDIO_DIR = "/kaggle/input/datasets/trngquangthi4139/mdd-challenge/MDD-Challenge-2025-training-set/audio_data/train"
    
    # --- ĐƯỜNG DẪN TẬP ĐÁNH GIÁ (PUBLIC TEST) ---
    PUBLIC_CSV = "/kaggle/input/datasets/trngquangthi4139/mdd-public-test-36/MDD-Challenge-2025-public-test/metadata/public_test_phones.csv"
    PUBLIC_AUDIO_DIR = "/kaggle/input/datasets/trngquangthi4139/mdd-public-test-36/MDD-Challenge-2025-public-test/audio_data/public_test"
    
    MODEL_CHECKPOINT = "nguyenvulebinh/wav2vec2-base-vietnamese-250h"
    BATCH_SIZE = 8
    EPOCHS = 60             
    LEARNING_RATE = 3e-5
    MAX_DURATION = 12.0
    DIAG_LOSS_TYPE = 'weighted_ce'

from transformers import Wav2Vec2Processor
processor = Wav2Vec2Processor.from_pretrained(Config.MODEL_CHECKPOINT)
print(f"🔥 Cấu hình liên kết Public Test thành công.")

preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

🔥 Cấu hình liên kết Public Test thành công.


In [ ]:
# =====================================================================
# CELL 3: HỢP NHẤT TỪ ĐIỂN PHONEME CHỐNG TRÙNG LẶP
# =====================================================================
import glob
import pandas as pd

df = pd.read_csv(Config.TRAIN_CSV)

# 1. Thu thập từ điển 1: Dùng .split() để bóc tách chính xác từng cụm âm vị cách nhau bằng khoảng trắng
all_transcripts = " ".join(df['transcript'].astype(str).tolist())
vocab_train_set = set(all_transcripts.split()) 

# 2. Thu thập từ điển 2: Lấy thêm âm vị từ file lexicon_vmd.txt của cuộc thi
lexicon_search = glob.glob("/kaggle/input/**/lexicon_vmd.txt", recursive=True)
LEXICON_PATH = lexicon_search[0] if len(lexicon_search) > 0 else os.path.join(Config.DATA_DIR, "lexicon_vmd.txt")
vocab_lexicon_set = set()
if os.path.exists(LEXICON_PATH):
    with open(LEXICON_PATH, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                for ph in parts[1:]:
                    vocab_lexicon_set.add(ph)
    print(f"📖 Tìm thấy {len(vocab_lexicon_set)} âm vị chuẩn trong Lexicon.")
else:
    print("⚠️ Không tìm thấy file lexicon, chỉ sử dụng từ điển từ tập Train.")

# 3. Hợp nhất hai nguồn âm vị, loại bỏ PAD/UNK hệ thống
fused_vocab = sorted(list(vocab_train_set.union(vocab_lexicon_set)))
fused_vocab = [tok for tok in fused_vocab if tok not in ["[PAD]", "[UNK]"]]

# Xây dựng bảng hash map chuyển đổi Phoneme thành ID mã hóa
char2idx = {token: idx for idx, token in enumerate(fused_vocab)}
char2idx["[PAD]"] = len(char2idx)
char2idx["[UNK]"] = len(char2idx)
diag2idx = {"C": 0, "S": 1, "D": 2, "I": 3, "[PAD]": 4}

print(f"📊 KẾT QUẢ HỢP NHẤT TỪ ĐIỂN KHÔNG TRÙNG LẶP:")
print(f"   ▶️ Tổng số lượng Token âm vị hệ thống sẽ học (Vocab Size): {len(char2idx)}")

# --- THUẬT TOÁN CĂN HÀNG PHỤC VỤ SINH NHÃN ĐÚNG BẢN CHẤT MDD ---
def _align_static(seq1, seq2):
    GAP = "<eps>"; MATCH = 1; MISMATCH = -1
    n, m = len(seq1), len(seq2)
    score = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): score[i][0] = -1 * i
    for j in range(n + 1): score[0][j] = -1 * j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            s = MATCH if seq1[j-1] == seq2[i-1] else (-1 if seq1[j-1] == GAP or seq2[i-1] == GAP else MISMATCH)
            score[i][j] = max(score[i-1][j-1] + s, score[i-1][j] - 1, score[i][j-1] - 1)
    align1, align2 = [], []
    i, j = m, n
    while i > 0 and j > 0:
        s = MATCH if seq1[j-1] == seq2[i-1] else (-1 if seq1[j-1] == GAP or seq2[i-1] == GAP else MISMATCH)
        if score[i][j] == score[i-1][j-1] + s:
            align1.append(seq1[j-1]); align2.append(seq2[i-1]); i -= 1; j -= 1
        elif score[i][j] == score[i][j-1] - 1:
            align1.append(seq1[j-1]); align2.append(GAP); j -= 1
        else:
            align1.append(GAP); align2.append(seq2[i-1]); i -= 1
    while j > 0: align1.append(seq1[j-1]); align2.append(GAP); j -= 1
    while i > 0: align1.append(GAP); align2.append(seq2[i-1]); i -= 1
    align1.reverse(); align2.reverse()
    return align1, align2

# 4. Hàm sinh nhãn chẩn đoán lỗi dựa trên thuật toán căn hàng chuẩn
def generate_diag_labels(row):
    c_phonemes = str(row['canonical']).split()
    t_phonemes = str(row['transcript']).split()
    
    # Căn hàng hai chuỗi tương tự như cách tính Metric của BTC
    ref_seq, human_seq = _align_static(c_phonemes, t_phonemes)
    
    ops = []
    for r, h in zip(ref_seq, human_seq):
        if r != "<eps>" and h == "<eps>": ops.append("D")
        elif r == "<eps>" and h != "<eps>": ops.append("I")
        elif r != h: ops.append("S")
        else: ops.append("C")
        
    # Lọc nhãn chẩn đoán: Chỉ giữ lại nhãn tương ứng với vị trí thực tế có trong chuỗi Transcript
    clean_labels = [op for op, h_char in zip(ops, human_seq) if h_char != "<eps>"]
    return " ".join(clean_labels)

df['diag_sequence'] = df.apply(generate_diag_labels, axis=1)

📖 Tìm thấy 122 âm vị chuẩn trong Lexicon.
📊 KẾT QUẢ HỢP NHẤT TỪ ĐIỂN KHÔNG TRÙNG LẶP:
   ▶️ Tổng số lượng Token âm vị hệ thống sẽ học (Vocab Size): 124


In [ ]:
# =====================================================================
# CELL 4: CUSTOM PYTORCH DATASET & DATALOADER
# =====================================================================
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import soundfile as sf
import torchaudio
from torch.utils.data import Dataset, DataLoader

# Hàm bổ trợ sinh nhãn tự động trực tiếp trên RAM cho tập Public Test
def _align_online(seq1, seq2):
    GAP = "<eps>"; MATCH = 1; MISMATCH = -1
    n, m = len(seq1), len(seq2)
    score = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): score[i][0] = -1 * i
    for j in range(n + 1): score[0][j] = -1 * j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            s = MATCH if seq1[j-1] == seq2[i-1] else (-1 if seq1[j-1] == GAP or seq2[i-1] == GAP else MISMATCH)
            score[i][j] = max(score[i-1][j-1] + s, score[i-1][j] - 1, score[i][j-1] - 1)
    align1, align2 = [], []
    i, j = m, n
    while i > 0 and j > 0:
        s = MATCH if seq1[j-1] == seq2[i-1] else (-1 if seq1[j-1] == GAP or seq2[i-1] == GAP else MISMATCH)
        if score[i][j] == score[i-1][j-1] + s:
            align1.append(seq1[j-1]); align2.append(seq2[i-1]); i -= 1; j -= 1
        elif score[i][j] == score[i][j-1] - 1:
            align1.append(seq1[j-1]); align2.append(GAP); j -= 1
        else:
            align1.append(GAP); align2.append(seq2[i-1]); i -= 1
    while j > 0: align1.append(seq1[j-1]); align2.append(GAP); j -= 1
    while i > 0: align1.append(GAP); align2.append(seq2[i-1]); i -= 1
    align1.reverse(); align2.reverse()
    
    # Sinh mảng thao tác lỗi op_rh
    ops = []
    for r, h in zip(align1, align2):
        if r != "<eps>" and h == "<eps>": ops.append("D")
        elif r == "<eps>" and h != "<eps>": ops.append("I")
        elif r != h: ops.append("S")
        else: ops.append("C")
        
    # Lọc bỏ các phần tử tương ứng với khoảng trống của ký tự chuẩn để thu được chuỗi diag_sequence thực tế
    diag_seq = [op for op, r in zip(ops, align1) if r != "<eps>"]
    return diag_seq

class SpeedPerturbation:
    def __init__(self, sample_rate=16000):
        self.sample_rate = sample_rate

    def __call__(self, audio_data):
        speed_factor = np.random.uniform(0.9, 1.1)
        if speed_factor == 1.0:  
            return audio_data
        if not isinstance(audio_data, torch.Tensor):
            audio_data = torch.FloatTensor(audio_data).unsqueeze(0)
        elif audio_data.dim() == 1:
            audio_data = audio_data.unsqueeze(0)
        sox_effects = [["speed", str(speed_factor)], ["rate", str(self.sample_rate)]]
        transformed_audio, _ = torchaudio.sox_effects.apply_effects_tensor(audio_data, self.sample_rate, sox_effects)
        return transformed_audio.squeeze(0).numpy()

class MDDAugmentedDataset(Dataset):
    def __init__(self, dataframe, audio_dir, char2idx, diag2idx, aug_prob=0.3, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.audio_dir = audio_dir  
        self.char2idx = char2idx
        self.diag2idx = diag2idx
        self.aug_prob = aug_prob
        self.is_train = is_train
        self.valid_chars = [c for c in char2idx.keys() if c not in ["[PAD]", "[UNK]"]]
        self.speed_perturb = SpeedPerturbation(sample_rate=16000)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        csv_path = str(row['path']).strip()
        
        filename = os.path.basename(csv_path)
        audio_path = os.path.join(self.audio_dir, filename)
        if not audio_path.endswith('.wav'):
            audio_path += '.wav'
            
        speech, sample_rate = sf.read(audio_path)
        
        # Giữ nguyên toàn bộ audio gốc
        
        if self.aug_prob > 0.0 and random.random() < self.aug_prob:
            speech = self.speed_perturb(speech)
            
        transcript_list = str(row['transcript']).split()
        canonical_list = str(row['canonical']).split()
        
        if 'diag_sequence' in row and pd.notna(row['diag_sequence']):
            diag_sequence = str(row['diag_sequence']).split()
        else:
            diag_sequence = _align_online(canonical_list, transcript_list)
            
        if len(diag_sequence) != len(canonical_list):
            diag_sequence = diag_sequence[:len(canonical_list)] + ["C"] * max(0, len(canonical_list) - len(diag_sequence))
        
        clean_target_asr = [self.char2idx.get(c, self.char2idx["[UNK]"]) for c in transcript_list]
        clean_canonical_asr = [self.char2idx.get(c, self.char2idx["[UNK]"]) for c in canonical_list]
        clean_target_diag = [self.diag2idx.get(d, self.diag2idx["[PAD]"]) for d in diag_sequence]
        
        # ĐẢM BẢO BATCH NÀO CŨNG CÓ LỖI PHÁT ÂM SAI
        if self.is_train and len(transcript_list) > 3:
            is_all_correct = all(d == "C" for d in diag_sequence)
            
            # Nếu câu gốc hoàn toàn đúng -> ÉP BUỘC gieo lỗi 100%. Nếu không, dựa theo aug_prob xúc xắc.
            if is_all_correct or (self.aug_prob > 0.0 and random.random() < self.aug_prob):
                num_mutations = random.choice([1, 2])
                mutation_indices = random.sample(range(len(transcript_list)), min(num_mutations, len(transcript_list)))
                for mutation_idx in mutation_indices:
                    aug_type = random.choice(["SUB", "DEL", "INS"])
                    if aug_type == "SUB" and len(self.valid_chars) > 0:
                        original_char = transcript_list[mutation_idx]
                        wrong_char = random.choice([c for c in self.valid_chars if c != original_char])
                        transcript_list[mutation_idx] = wrong_char
                        if mutation_idx < len(diag_sequence): diag_sequence[mutation_idx] = "S"
                    elif aug_type == "DEL" and len(transcript_list) > mutation_idx:
                        transcript_list.pop(mutation_idx)
                        if mutation_idx < len(diag_sequence): diag_sequence[mutation_idx] = "D"
                        break 
                    elif aug_type == "INS" and len(self.valid_chars) > 0:
                        extra_char = random.choice(self.valid_chars)
                        transcript_list.insert(mutation_idx, extra_char)
                        if mutation_idx < len(diag_sequence): diag_sequence.insert(mutation_idx, "I")
                        break

        target_asr = [self.char2idx.get(c, self.char2idx["[UNK]"]) for c in transcript_list]
        target_diag = [self.diag2idx.get(d, self.diag2idx["[PAD]"]) for d in diag_sequence]
        
        return {
            "speech": speech,
            "target_asr": target_asr,
            "target_diag": target_diag,
            "clean_target_asr": clean_target_asr,    
            "clean_canonical_asr": clean_canonical_asr,   
            "clean_target_diag": clean_target_diag   
        }

def collate_fn(batch):
    speech_list = [item["speech"] for item in batch]
    inputs = processor(speech_list, sampling_rate=16000, padding=True, return_attention_mask=True, return_tensors="pt")
    
    asr_padded = nn.utils.rnn.pad_sequence([torch.tensor(item["target_asr"]) for item in batch], batch_first=True, padding_value=char2idx["[PAD]"])
    diag_padded = nn.utils.rnn.pad_sequence([torch.tensor(item["target_diag"]) for item in batch], batch_first=True, padding_value=diag2idx["[PAD]"])
    asr_lengths = torch.tensor([len(item["target_asr"]) for item in batch], dtype=torch.long)
    
    clean_asr_padded = nn.utils.rnn.pad_sequence([torch.tensor(item["clean_target_asr"]) for item in batch], batch_first=True, padding_value=char2idx["[PAD]"])
    clean_canonical_padded = nn.utils.rnn.pad_sequence([torch.tensor(item["clean_canonical_asr"]) for item in batch], batch_first=True, padding_value=char2idx["[PAD]"])
    clean_diag_padded = nn.utils.rnn.pad_sequence([torch.tensor(item["clean_target_diag"]) for item in batch], batch_first=True, padding_value=diag2idx["[PAD]"])
    
    return {
        "input_values": inputs.input_values,
        "attention_mask": inputs.attention_mask,
        "targets_asr": asr_padded,
        "targets_diag": diag_padded,
        "asr_lengths": asr_lengths,
        "clean_targets_asr": clean_asr_padded, 
        "clean_canonical_asr": clean_canonical_padded,   
        "clean_targets_diag": clean_diag_padded      
    }

# =====================================================================
# KHỞI TẠO DATALOADER MỞ RỘNG DATA
# =====================================================================
print("📊 Tiến hành tải dữ liệu liên kết động...")

df_train_raw = pd.read_csv(Config.TRAIN_CSV)
df_train_raw['phone_count'] = df_train_raw['transcript'].apply(lambda x: len(str(x).split()))

# NỚI RỘNG LÊN TOÀN BỘ CÂU DÀI TRÊN TẬP TRAIN (là lấy hết)
train_df_cleaned = df_train_raw[df_train_raw['phone_count'] <= 1000].copy()
print(f" -> Tập Train sạch (Mở rộng): {len(train_df_cleaned)} dòng.")

public_test_df = pd.read_csv(Config.PUBLIC_CSV)
print(f" -> Tập Public Test (Dùng để Val): {len(public_test_df)} dòng.")

train_dataset = MDDAugmentedDataset(train_df_cleaned, Config.TRAIN_AUDIO_DIR, char2idx, diag2idx, aug_prob=0.3, is_train=True)
val_dataset = MDDAugmentedDataset(public_test_df, Config.PUBLIC_AUDIO_DIR, char2idx, diag2idx, aug_prob=0.0, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
print("✅ Hoàn thành cấu hình Dataloader.")

📊 Tiến hành tải dữ liệu liên kết động...
 -> Tập Train sạch (Mở rộng): 3180 dòng.
 -> Tập Public Test (Dùng để Val): 718 dòng.
✅ Hoàn thành cấu hình Dataloader.


In [ ]:
# =====================================================================
# CELL 5: KIẾN TRÚC MÔ HÌNH MULTITASK 
# =====================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import Wav2Vec2Model

class FusedCrossAttention(nn.Module):
    def __init__(self, hidden_dim, num_classes_asr):
        super().__init__()
        self.asr_projection = nn.Linear(num_classes_asr, hidden_dim)
        self.query = nn.Linear(hidden_dim, hidden_dim)
        self.key = nn.Linear(hidden_dim, hidden_dim)
        self.value = nn.Linear(hidden_dim, hidden_dim)
        self.scale = torch.sqrt(torch.FloatTensor([hidden_dim]))

    def forward(self, sequence_output, asr_logits):
        asr_feat = self.asr_projection(asr_logits) # [B, T, Hidden]
        
        Q = self.query(asr_feat)             # Query từ nhận diện âm vị
        K = self.key(sequence_output)        # Key từ đặc trưng gốc âm thanh
        V = self.value(sequence_output)      # Value từ đặc trưng gốc âm thanh
        
        attn_energy = torch.bmm(Q, K.transpose(1, 2)) / self.scale.to(sequence_output.device)
        attn_weights = torch.softmax(attn_energy, dim=-1)
        fused_context = torch.bmm(attn_weights, V)
        
        return fused_context + sequence_output

class MultitaskMDDModel(nn.Module):
    def __init__(self, model_checkpoint, num_classes_asr, num_classes_diag, criterion=None):
        super().__init__()
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(model_checkpoint)
        hidden_dim = self.wav2vec2.config.hidden_size
        self.wav2vec2.feature_extractor._freeze_parameters()
        
        self.asr_head = nn.Linear(hidden_dim, num_classes_asr)
        self.fused_attention = FusedCrossAttention(hidden_dim, num_classes_asr)
        
        # Bổ sung Bi-LSTM để tăng sức mạnh mô hình hóa chuỗi lỗi cho Diagnosis Head
        self.diag_lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim // 2,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        self.diagnosis_head = nn.Linear(hidden_dim, num_classes_diag)
        
        # Nhận thực thể Loss Engine từ bên ngoài
        self.criterion = criterion 
        
    def forward(self, input_values, attention_mask=None, targets_asr=None, targets_diag=None, asr_lengths=None, epoch=0):
        outputs = self.wav2vec2(input_values, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        
        # Nhánh 1: Dự đoán thô ASR trước
        asr_logits = self.asr_head(sequence_output)
        
        # Nhánh 2: Fused thông tin ASR trợ lực cho nhánh Diagnosis
        fused_features = self.fused_attention(sequence_output, asr_logits.detach()) 
        
        # Đi qua tầng Bi-LSTM Context
        lstm_out, _ = self.diag_lstm(fused_features)
        diag_logits = self.diagnosis_head(lstm_out)
        
        # SONG SONG HOÀN TOÀN TRÊN MULTI-GPU:
        if self.criterion is not None and targets_asr is not None:
            features_lengths = torch.sum(attention_mask, dim=-1)
            input_lengths = self.wav2vec2._get_feat_extract_output_lengths(features_lengths)
            
            total_loss, loss_per, loss_diag = self.criterion(
                asr_logits=asr_logits, 
                diag_logits=diag_logits, 
                targets_asr=targets_asr, 
                targets_diag=targets_diag, 
                asr_lengths=asr_lengths, 
                input_lengths=input_lengths, 
                current_epoch=epoch
            )
            return total_loss, asr_logits, diag_logits
            
        return asr_logits, diag_logits

print("🏗️ Đã nạp cấu trúc mô hình nâng cấp Bi-LSTM v5.0.")

🏗️ Đã nạp cấu trúc mô hình nâng cấp Bi-LSTM v5.0.


In [ ]:
# =====================================================================
# CELL 6: ADAPTIVE LOSS ENGINE & ENGINE CONFIGURATION
# =====================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import os

class MDDUndirectedLoss(nn.Module):
    def __init__(self, char_pad_idx, diag_pad_idx, total_epochs=30, label_smoothing=0.05):
        super().__init__()
        self.asr_criterion = nn.CTCLoss(blank=char_pad_idx, zero_infinity=True)
        self.diag_pad_idx = diag_pad_idx
        self.label_smoothing = label_smoothing
        self.total_epochs = total_epochs
        
        # [C, D, I, S, PAD] -> Giữ nguyên bộ trọng số phạt nặng cấu trúc lỗi chẩn đoán
        class_weights = torch.tensor([1.0, 3.5, 4.0, 4.0, 0.0], dtype=torch.float)
        self.register_buffer('class_weights', class_weights)
        
        self.ce_criterion = nn.CrossEntropyLoss(
            weight=self.class_weights, 
            ignore_index=self.diag_pad_idx, 
            label_smoothing=self.label_smoothing
        )
        self.log_vars = nn.Parameter(torch.zeros(2))

    def forward(self, asr_logits, diag_logits, targets_asr, targets_diag, asr_lengths, input_lengths, current_epoch=0):
        device = asr_logits.device
        if self.ce_criterion.weight.device != device:
            self.ce_criterion.weight = self.class_weights.to(device)

        # 1. CTC Loss (Nhánh ASR)
        asr_log_probs = F.log_softmax(asr_logits, dim=-1).transpose(0, 1)
        loss_per = self.asr_criterion(asr_log_probs, targets_asr, input_lengths, asr_lengths)
        
        # 2. Đồng bộ chuỗi chẩn đoán
        batch_size, logit_seq_len, num_classes = diag_logits.size()
        target_seq_len = targets_diag.size(1)
        if logit_seq_len != target_seq_len:
            diag_logits_permuted = diag_logits.transpose(1, 2) 
            diag_logits_scaled = F.interpolate(
                diag_logits_permuted, 
                size=target_seq_len, 
                mode='linear', 
                align_corners=False
            )
            diag_logits = diag_logits_scaled.transpose(1, 2)
        
        # 3. Phẳng mảng tính Loss
        flat_diag_logits = diag_logits.reshape(-1, num_classes)
        flat_targets_diag = targets_diag.reshape(-1)
        loss_diag = self.ce_criterion(flat_diag_logits, flat_targets_diag)
        
        # --- CHIẾN LƯỢC ĐÒN BẨY ALPHA ĐƯỢC ĐẨY LÊN HƠN NỮA ---
        log_vars_clamped = torch.clamp(self.log_vars.to(device), min=-2.0, max=5.0)
        precision0 = torch.exp(-log_vars_clamped[0])
        precision1 = torch.exp(-log_vars_clamped[1])
        
        # Đẩy đòn bẩy alpha lên 7.5 (Tăng từ 5.0) để tạo lực ép cưỡng bức, triệt tiêu DER bằng mọi giá
        alpha = 7.5 
        
        weighted_loss_per = precision0 * loss_per + 0.5 * log_vars_clamped[0]
        weighted_loss_diag = (precision1 * alpha) * loss_diag + 0.5 * log_vars_clamped[1]
        
        total_loss = weighted_loss_per + weighted_loss_diag
        if total_loss.item() < 0:
            total_loss = loss_per + (alpha * loss_diag)
            
        return total_loss, loss_per.detach(), loss_diag.detach()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PHASE2_EPOCHS = 30 

criterion = MDDUndirectedLoss(
    char_pad_idx=char2idx["[PAD]"], 
    diag_pad_idx=diag2idx["[PAD]"], 
    total_epochs=PHASE2_EPOCHS,
    label_smoothing=0.05
)
criterion = criterion.to(device)

model = MultitaskMDDModel(Config.MODEL_CHECKPOINT, len(char2idx), len(diag2idx), criterion=criterion)

# 🔄 LOAD TỪ FILE FINETUNE ĐỈNH KỶ LỤC CŨ (EPOCH 3) ĐỂ KẾ THỪA KHÔNG GIAN VECTOR
PATH_TO_BEST = "/kaggle/input/notebooks/phuonglam26/mtl-fa-4-1/best_mdd_model.pt"
if os.path.exists(PATH_TO_BEST):
    print(f"🔄 Đang tải trọng số từ đỉnh kỷ lục cũ (Epoch 3): {PATH_TO_BEST}...")
    state_dict = torch.load(PATH_TO_BEST, map_location=device)
    model.load_state_dict(state_dict, strict=True)
    print("✅ Đã khôi phục hoàn tất Checkpoint nền tảng thành công!")
else:
    print(f"⚠️ Không tìm thấy file {PATH_TO_BEST}! Vui lòng kiểm tra lại đường dẫn.")

if torch.cuda.device_count() > 1:
    print(f"🚀 KÍCH HOẠT THÀNH CÔNG: Chạy song song {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)
model = model.to(device)

# --- THIẾT LẬP CẤU HÌNH OPTIMIZER TÁCH KHỐI CÂN BẰNG ---
raw_model = model.module if isinstance(model, nn.DataParallel) else model

backbone_params = raw_model.wav2vec2.parameters()
diagnosis_params = (
    list(raw_model.asr_head.parameters()) + 
    list(raw_model.fused_attention.parameters()) + 
    list(raw_model.diag_lstm.parameters()) + 
    list(raw_model.diagnosis_head.parameters()) +
    list(raw_model.criterion.parameters())
)

# Hạ LR Backbone xuống 1.5e-6 (Siêu mịn) để nắn chỉnh không gian vector chuyển động rất khẽ
optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': 1.5e-6},       
    {'params': diagnosis_params, 'lr': 4e-5}       
], weight_decay=1e-4)

# --- NỚI RỘNG PATIENCE CHO SCHEDULER THEO DÕI NỀN TẢNG THỰC TẾ ---
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=3      
)

print(f"✅ Đã tái cấu hình bộ tối ưu chuyên dụng KHÓA CHỌN LỌC GIẢM DER. Sẵn sàng khởi chạy.")

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: nguyenvulebinh/wav2vec2-base-vietnamese-250h
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 
lm_head.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔄 Đang tải trọng số từ đỉnh kỷ lục cũ (Epoch 3): /kaggle/input/notebooks/phuonglam26/mtl-fa-4-1/best_mdd_model.pt...


model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

✅ Đã khôi phục hoàn tất Checkpoint nền tảng thành công!
🚀 KÍCH HOẠT THÀNH CÔNG: Chạy song song 2 GPUs!
✅ Đã tái cấu hình bộ tối ưu chuyên dụng KHÓA CHỌN LỌC GIẢM DER. Sẵn sàng khởi chạy.


In [ ]:
# =====================================================================
# CELL 7: TRAINING & VALIDATION LOOP WITH SELECTIVE FREEZING STRATEGY
# =====================================================================
import numpy as np
import torch

def _align(seq1, seq2):
    GAP = "<eps>"; MATCH = 1; MISMATCH = -1
    n, m = len(seq1), len(seq2)
    score = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): score[i][0] = -1 * i
    for j in range(n + 1): score[0][j] = -1 * j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            s = MATCH if seq1[j-1] == seq2[i-1] else (-1 if seq1[j-1] == GAP or seq2[i-1] == GAP else MISMATCH)
            score[i][j] = max(score[i-1][j-1] + s, score[i-1][j] - 1, score[i][j-1] - 1)
    align1, align2 = [], []
    i, j = m, n
    while i > 0 and j > 0:
        s = MATCH if seq1[j-1] == seq2[i-1] else (-1 if seq1[j-1] == GAP or seq2[i-1] == GAP else MISMATCH)
        if score[i][j] == score[i-1][j-1] + s:
            align1.append(seq1[j-1]); align2.append(seq2[i-1]); i -= 1; j -= 1
        elif score[i][j] == score[i][j-1] - 1:
            align1.append(seq1[j-1]); align2.append(GAP); j -= 1
        else:
            align1.append(GAP); align2.append(seq2[i-1]); i -= 1
    while j > 0: align1.append(seq1[j-1]); align2.append(GAP); j -= 1
    while i > 0: align1.append(GAP); align2.append(seq2[i-1]); i -= 1
    align1.reverse(); align2.reverse()
    return align1, align2

def _ops(aligned1, aligned2):
    return ["D" if r != "<eps>" and h == "<eps>" else "I" if r == "<eps>" and h != "<eps>" else "S" if r != h else "C" for r, h in zip(aligned1, aligned2)]

def calculate_mdd_metrics_standard(all_preds_asr, all_targets_asr, all_canonical_asr, char_pad_idx, idx2char):
    cor_cor = cor_nocor = sub_sub = sub_sub1 = sub_nosub = 0
    ins_ins = ins_ins1 = ins_noins = del_del = del_del1 = del_nodel = 0
    total_sub = total_del = total_ins = total_ref_len = 0 

    for pred_ids, human_ids, ref_ids in zip(all_preds_asr, all_targets_asr, all_canonical_asr):
        pred_clean = []
        prev = None
        for x in pred_ids:
            if x != char_pad_idx and x != prev: pred_clean.append(x)
            prev = x
            
        human_clean = [x for x in human_ids if x != char_pad_idx]
        ref_clean = [x for x in ref_ids if x != char_pad_idx]
        if len(human_clean) == 0: continue

        our_seq_in = [idx2char.get(i, "<unk>") for i in pred_clean]
        human_seq_in = [idx2char.get(i, "<unk>") for i in human_clean]
        ref_seq_in = [idx2char.get(i, "<unk>") for i in ref_clean]

        ref_seq, human_seq = _align(ref_seq_in, human_seq_in)
        op_rh = _ops(ref_seq, human_seq)
        human_seq2, our_seq2 = _align(human_seq_in, our_seq_in)
        op_ho = _ops(human_seq2, our_seq2)
        ref_seq3, our_seq3 = _align(ref_seq_in, our_seq_in)
        op_ro = _ops(ref_seq3, our_seq3)
        
        sub, del_, ins, cor = op_ho.count("S"), op_ho.count("D"), op_ho.count("I"), op_ho.count("C")
        total_sub += sub; total_del += del_; total_ins += ins; total_ref_len += (sub + del_ + cor)
        
        flag = 0
        for i in range(len(ref_seq)):
            if ref_seq[i] == "<eps>": continue
            while flag < len(ref_seq3) and ref_seq3[flag] == "<eps>": flag += 1
            if flag < len(ref_seq3) and ref_seq[i] == ref_seq3[flag]:
                if op_rh[i] == "D" and op_ro[flag] == "D": del_del += 1
                elif op_rh[i] == "D" and op_ro[flag] != "D" and op_ro[flag] != "C": del_del1 += 1
                elif op_rh[i] == "D" and op_ro[flag] != "D" and op_ro[flag] == "C": del_nodel += 1
                flag += 1

        flag = 0
        for i in range(len(human_seq)):
            if human_seq[i] == "<eps>": continue
            while flag < len(human_seq2) and human_seq2[flag] == "<eps>": flag += 1
            if flag < len(human_seq2) and human_seq[i] == human_seq2[flag]:
                if op_rh[i] == "C" and op_ho[flag] == "C": cor_cor += 1
                elif op_rh[i] == "C" and op_ho[flag] != "C": cor_nocor += 1
                elif op_rh[i] == "S" and op_ho[flag] == "C": sub_sub += 1
                elif op_rh[i] == "S" and op_ho[flag] != "C" and ref_seq[i] != our_seq2[flag]: sub_sub1 += 1
                elif op_rh[i] == "S" and op_ho[flag] != "C" and ref_seq[i] == our_seq2[flag]: sub_nosub += 1
                elif op_rh[i] == "I" and op_ho[flag] == "C": ins_ins += 1
                elif op_rh[i] == "I" and op_ho[flag] != "C" and op_ho[flag] != "D": ins_ins1 += 1
                elif op_rh[i] == "I" and op_ho[flag] != "C" and op_ho[flag] == "D": ins_noins += 1
                flag += 1
                
    TR = sub_sub + sub_sub1 + del_del + del_del1 + ins_ins + ins_ins1
    precision = TR / (TR + cor_nocor) if (TR + cor_nocor) > 0 else 0.0
    recall = TR / (TR + (sub_nosub + ins_noins + del_nodel)) if (TR + (sub_nosub + ins_noins + del_nodel)) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    per = (total_sub + total_del + total_ins) / total_ref_len if total_ref_len > 0 else 1.0
    der = (sub_sub1 + del_del1 + ins_ins1) / TR if TR > 0 else 0.0
    
    return f1, der, per, (0.5 * f1 + 0.4 * (1.0 - der) + 0.1 * (1.0 - per))

idx2char = {v: k for k, v in char2idx.items()}
start_epoch = 0
best_challenge_score = 0.5373  # Ghi nhận đỉnh kỷ lục thực tế đạt được ở Epoch 3 làm mốc bảo hiểm

print(f"🆕 Kích hoạt vòng lặp Fine-Tuning bứt phá DER bằng cơ chế ĐÓNG BĂNG CHỌN LỌC...")
for epoch in range(start_epoch, PHASE2_EPOCHS):
    
    # 🔓 [SELECTIVE FREEZING STRATEGY] Thiết quân luật cho Wav2Vec2
    raw_model = model.module if isinstance(model, nn.DataParallel) else model
    
    # Bước A: Khóa cứng toàn bộ tham số xương sống trước
    for param in raw_model.wav2vec2.parameters():
        param.requires_grad = False
        
    # Bước B: Chỉ giải phóng đúng 2 tầng Transformer cuối cùng ở phần Encoder của Wav2Vec2
    for layer in raw_model.wav2vec2.encoder.layers[-2:]:
        for param in layer.parameters():
            param.requires_grad = True
            
    print(f"🔒 [Selective Frozen] Đã khóa tầng thấp. Chỉ mở khóa 2 tầng Transformer cuối tại Epoch {epoch+1}.")

    # --- 1. TIẾN TRÌNH HUẤN LUYỆN (TRAINING MODE) ---
    model.train()
    running_loss = 0.0
    train_preds_asr, train_targets_asr, train_canonical_asr = [], [], []
    
    for batch_idx, batch in enumerate(train_loader):
        input_values = batch["input_values"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets_asr = batch["targets_asr"].to(device)      
        targets_diag = batch["targets_diag"].to(device)    
        asr_lengths = batch["asr_lengths"].to(device)
        
        optimizer.zero_grad()
        
        total_loss, asr_logits, diag_logits = model(
            input_values=input_values, 
            attention_mask=attention_mask,
            targets_asr=targets_asr,
            targets_diag=targets_diag,
            asr_lengths=asr_lengths,
            epoch=epoch
        )
        
        loss = total_loss.mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        running_loss += loss.item()
        
        with torch.no_grad():
            train_preds_asr.extend(asr_logits.detach().argmax(dim=-1).cpu().numpy())
            train_targets_asr.extend(batch["clean_targets_asr"].numpy())
            train_canonical_asr.extend(batch["clean_canonical_asr"].numpy())
        
        if batch_idx % 20 == 0:
            print(f"Fine-Tuning Epoch [{epoch+1}/{PHASE2_EPOCHS}] | Batch [{batch_idx}/{len(train_loader)}] | Loss: {loss.item():.4f}")
            
    # Lấy thông tin LR hiện thời từ Optimizer
    current_lrs = [group['lr'] for group in optimizer.param_groups]
    backbone_lr = current_lrs[0]
    diag_lr = current_lrs[1] if len(current_lrs) > 1 else current_lrs[0]
        
    # --- 2. TIẾN TRÌNH ĐÁNH GIÁ (VALIDATION MODE) ---
    model.eval()
    val_total_loss = 0.0
    val_preds_asr, val_targets_asr, val_canonical_asr = [], [], []
    
    with torch.no_grad():
        for batch in val_loader:
            input_values = batch["input_values"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            targets_asr = batch["targets_asr"].to(device)
            targets_diag = batch["targets_diag"].to(device)
            asr_lengths = batch["asr_lengths"].to(device)
            
            total_loss, asr_logits, diag_logits = model(
                input_values=input_values, 
                attention_mask=attention_mask,
                targets_asr=targets_asr,
                targets_diag=targets_diag,
                asr_lengths=asr_lengths,
                epoch=epoch
            )
            
            val_total_loss += total_loss.mean().item()
            
            val_preds_asr.extend(asr_logits.argmax(dim=-1).cpu().numpy())
            val_targets_asr.extend(batch["clean_targets_asr"].numpy()) 
            val_canonical_asr.extend(batch["clean_canonical_asr"].numpy())
            
    epoch_train_loss = running_loss / len(train_loader)
    epoch_val_loss = val_total_loss / len(val_loader)
    
    train_f1, train_der, train_per, train_score = calculate_mdd_metrics_standard(
        train_preds_asr, train_targets_asr, train_canonical_asr, char_pad_idx=char2idx["[PAD]"], idx2char=idx2char
    )
    val_f1, val_der, val_per, val_score = calculate_mdd_metrics_standard(
        val_preds_asr, val_targets_asr, val_canonical_asr, char_pad_idx=char2idx["[PAD]"], idx2char=idx2char
    )
    
    # CẬP NHẬT SCHEDULER: Truyền điểm Val thực tế
    scheduler.step(val_score)
    
    print(f"\n====================== KẾT THÚC FINE-TUNING EPOCH {epoch+1} ======================")
    print(f"📉 Loss Train: {epoch_train_loss:.4f} | Val: {epoch_val_loss:.4f}")
    print(f"🔧 LR Backbone: {backbone_lr:.8f} | LR Diagnosis: {diag_lr:.8f}")
    print(f"🚀 VAL CHALLENGE SCORE: {val_score:.4f} (Đỉnh kỷ lục cũ: {best_challenge_score:.4f})")
    print(f"   1. F1-Score: {val_f1:.4f} | 2. DER: {val_der*100:.2f}% | 3. PER: {val_per*100:.2f}%")
    print(f"======================================================================\n")
    
    raw_model_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()

    if val_score > best_challenge_score:
        best_challenge_score = val_score
        torch.save(raw_model_state, "best_mdd_model.pt")
        print(f"🔥 [NEW RECORD DETECTED!] Điểm số lập đỉnh mới đạt: {val_score:.4f}! Đã cập nhật file: best_mdd_model.pt\n")

🆕 Kích hoạt vòng lặp Fine-Tuning bứt phá DER bằng cơ chế ĐÓNG BĂNG CHỌN LỌC...
🔒 [Selective Frozen] Đã khóa tầng thấp. Chỉ mở khóa 2 tầng Transformer cuối tại Epoch 1.


/tmp/ipykernel_22/4172359546.py:61: UserWarning: torchaudio.sox_effects.sox_effects.apply_effects_tensor has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  transformed_audio, _ = torchaudio.sox_effects.apply_effects_tensor(audio_data, self.sample_rate, sox_effects)
/tmp/ipykernel_22/4172359546.py:61: UserWarning: torchaudio.sox_effects.sox_effects.apply_effects_tensor has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  transformed_audio, _ = torchaudio.sox_effects.apply_effects_tensor(audio_data, self.sample_rate, sox_effects)
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:

Fine-Tuning Epoch [1/30] | Batch [0/398] | Loss: 4.7434
Fine-Tuning Epoch [1/30] | Batch [20/398] | Loss: 5.1896
Fine-Tuning Epoch [1/30] | Batch [40/398] | Loss: 4.2105
Fine-Tuning Epoch [1/30] | Batch [60/398] | Loss: 4.7485
Fine-Tuning Epoch [1/30] | Batch [80/398] | Loss: 3.9668
Fine-Tuning Epoch [1/30] | Batch [100/398] | Loss: 4.9232
Fine-Tuning Epoch [1/30] | Batch [120/398] | Loss: 3.9818
Fine-Tuning Epoch [1/30] | Batch [140/398] | Loss: 4.2765
Fine-Tuning Epoch [1/30] | Batch [160/398] | Loss: 4.7706
Fine-Tuning Epoch [1/30] | Batch [180/398] | Loss: 4.0885
Fine-Tuning Epoch [1/30] | Batch [200/398] | Loss: 4.8928
Fine-Tuning Epoch [1/30] | Batch [220/398] | Loss: 4.3835
Fine-Tuning Epoch [1/30] | Batch [240/398] | Loss: 3.7510
Fine-Tuning Epoch [1/30] | Batch [260/398] | Loss: 4.5471
Fine-Tuning Epoch [1/30] | Batch [280/398] | Loss: 3.9209
Fine-Tuning Epoch [1/30] | Batch [300/398] | Loss: 4.1597
Fine-Tuning Epoch [1/30] | Batch [320/398] | Loss: 4.4044
Fine-Tuning Epoch [1

/tmp/ipykernel_22/4172359546.py:61: UserWarning: torchaudio.sox_effects.sox_effects.apply_effects_tensor has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  transformed_audio, _ = torchaudio.sox_effects.apply_effects_tensor(audio_data, self.sample_rate, sox_effects)
/tmp/ipykernel_22/4172359546.py:61: UserWarning: torchaudio.sox_effects.sox_effects.apply_effects_tensor has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  transformed_audio, _ = torchaudio.sox_effects.apply_effects_tensor(audio_data, self.sample_rate, sox_effects)


Fine-Tuning Epoch [2/30] | Batch [0/398] | Loss: 4.1372
Fine-Tuning Epoch [2/30] | Batch [20/398] | Loss: 4.4614
Fine-Tuning Epoch [2/30] | Batch [40/398] | Loss: 4.3148
Fine-Tuning Epoch [2/30] | Batch [60/398] | Loss: 4.1538
Fine-Tuning Epoch [2/30] | Batch [80/398] | Loss: 3.9604
Fine-Tuning Epoch [2/30] | Batch [100/398] | Loss: 4.4551
Fine-Tuning Epoch [2/30] | Batch [120/398] | Loss: 4.7133
Fine-Tuning Epoch [2/30] | Batch [140/398] | Loss: 3.6931
Fine-Tuning Epoch [2/30] | Batch [160/398] | Loss: 3.8022
Fine-Tuning Epoch [2/30] | Batch [180/398] | Loss: 3.9199
Fine-Tuning Epoch [2/30] | Batch [200/398] | Loss: 4.1280
Fine-Tuning Epoch [2/30] | Batch [220/398] | Loss: 3.9747
Fine-Tuning Epoch [2/30] | Batch [240/398] | Loss: 4.1656
Fine-Tuning Epoch [2/30] | Batch [260/398] | Loss: 4.4764
Fine-Tuning Epoch [2/30] | Batch [280/398] | Loss: 4.1652
Fine-Tuning Epoch [2/30] | Batch [300/398] | Loss: 4.6261
Fine-Tuning Epoch [2/30] | Batch [320/398] | Loss: 5.4844
Fine-Tuning Epoch [2

...

Fine-Tuning Epoch [28/30] | Batch [0/398] | Loss: 3.3164
Fine-Tuning Epoch [28/30] | Batch [20/398] | Loss: 3.4750
Fine-Tuning Epoch [28/30] | Batch [40/398] | Loss: 3.3974
Fine-Tuning Epoch [28/30] | Batch [60/398] | Loss: 3.9410
Fine-Tuning Epoch [28/30] | Batch [80/398] | Loss: 3.3457
Fine-Tuning Epoch [28/30] | Batch [100/398] | Loss: 4.0355
Fine-Tuning Epoch [28/30] | Batch [120/398] | Loss: 3.5892
Fine-Tuning Epoch [28/30] | Batch [140/398] | Loss: 3.4747
Fine-Tuning Epoch [28/30] | Batch [160/398] | Loss: 3.6676
Fine-Tuning Epoch [28/30] | Batch [180/398] | Loss: 3.7453
Fine-Tuning Epoch [28/30] | Batch [200/398] | Loss: 3.6810
Fine-Tuning Epoch [28/30] | Batch [220/398] | Loss: 3.7751
Fine-Tuning Epoch [28/30] | Batch [240/398] | Loss: 3.6954
Fine-Tuning Epoch [28/30] | Batch [260/398] | Loss: 3.8630
Fine-Tuning Epoch [28/30] | Batch [280/398] | Loss: 3.8942
Fine-Tuning Epoch [28/30] | Batch [300/398] | Loss: 3.4834
Fine-Tuning Epoch [28/30] | Batch [320/398] | Loss: 3.7331
Fin

/tmp/ipykernel_22/4172359546.py:61: UserWarning: torchaudio.sox_effects.sox_effects.apply_effects_tensor has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  transformed_audio, _ = torchaudio.sox_effects.apply_effects_tensor(audio_data, self.sample_rate, sox_effects)
/tmp/ipykernel_22/4172359546.py:61: UserWarning: torchaudio.sox_effects.sox_effects.apply_effects_tensor has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  transformed_audio, _ = torchaudio.sox_effects.apply_effects_tensor(audio_data, self.sample_rate, sox_effects)


Fine-Tuning Epoch [29/30] | Batch [0/398] | Loss: 3.7470
Fine-Tuning Epoch [29/30] | Batch [20/398] | Loss: 3.6321
Fine-Tuning Epoch [29/30] | Batch [40/398] | Loss: 3.5130
Fine-Tuning Epoch [29/30] | Batch [60/398] | Loss: 3.3133
Fine-Tuning Epoch [29/30] | Batch [80/398] | Loss: 3.7782
Fine-Tuning Epoch [29/30] | Batch [100/398] | Loss: 3.4857
Fine-Tuning Epoch [29/30] | Batch [120/398] | Loss: 3.6189
Fine-Tuning Epoch [29/30] | Batch [140/398] | Loss: 3.5094
Fine-Tuning Epoch [29/30] | Batch [160/398] | Loss: 3.2540
Fine-Tuning Epoch [29/30] | Batch [180/398] | Loss: 4.4977
Fine-Tuning Epoch [29/30] | Batch [200/398] | Loss: 3.7425
Fine-Tuning Epoch [29/30] | Batch [220/398] | Loss: 3.9418
Fine-Tuning Epoch [29/30] | Batch [240/398] | Loss: 3.5144
Fine-Tuning Epoch [29/30] | Batch [260/398] | Loss: 3.0511
Fine-Tuning Epoch [29/30] | Batch [280/398] | Loss: 3.4160
Fine-Tuning Epoch [29/30] | Batch [300/398] | Loss: 3.5907
Fine-Tuning Epoch [29/30] | Batch [320/398] | Loss: 3.4502
Fin

/tmp/ipykernel_22/4172359546.py:61: UserWarning: torchaudio.sox_effects.sox_effects.apply_effects_tensor has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  transformed_audio, _ = torchaudio.sox_effects.apply_effects_tensor(audio_data, self.sample_rate, sox_effects)
/tmp/ipykernel_22/4172359546.py:61: UserWarning: torchaudio.sox_effects.sox_effects.apply_effects_tensor has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  transformed_audio, _ = torchaudio.sox_effects.apply_effects_tensor(audio_data, self.sample_rate, sox_effects)


Fine-Tuning Epoch [30/30] | Batch [0/398] | Loss: 3.9586
Fine-Tuning Epoch [30/30] | Batch [20/398] | Loss: 3.5892
Fine-Tuning Epoch [30/30] | Batch [40/398] | Loss: 3.7778
Fine-Tuning Epoch [30/30] | Batch [60/398] | Loss: 3.8730
Fine-Tuning Epoch [30/30] | Batch [80/398] | Loss: 3.4017
Fine-Tuning Epoch [30/30] | Batch [100/398] | Loss: 3.3072
Fine-Tuning Epoch [30/30] | Batch [120/398] | Loss: 3.4740
Fine-Tuning Epoch [30/30] | Batch [140/398] | Loss: 3.5228
Fine-Tuning Epoch [30/30] | Batch [160/398] | Loss: 3.4501
Fine-Tuning Epoch [30/30] | Batch [180/398] | Loss: 4.1362
Fine-Tuning Epoch [30/30] | Batch [200/398] | Loss: 3.3507
Fine-Tuning Epoch [30/30] | Batch [220/398] | Loss: 3.5507
Fine-Tuning Epoch [30/30] | Batch [240/398] | Loss: 4.2177
Fine-Tuning Epoch [30/30] | Batch [260/398] | Loss: 3.3834
Fine-Tuning Epoch [30/30] | Batch [280/398] | Loss: 3.4603
Fine-Tuning Epoch [30/30] | Batch [300/398] | Loss: 4.2124
Fine-Tuning Epoch [30/30] | Batch [320/398] | Loss: 3.4137
Fin

In [ ]:
# =====================================================================
# 🏁 CELL 8: SUY DIỄN TRÊN PRIVATE TEST (ĐƯỜNG DẪN TRỰC TIẾP)
# =====================================================================
import os
import gc
import torch
import pandas as pd
import numpy as np
import soundfile as sf
from tqdm import tqdm
import torch.nn as nn

# =====================================================================
# 🛠️ 1. BẠN ĐIỀN LINK TRỰC TIẾP CỦA TẬP PRIVATE TEST VÀO ĐÂY
# =====================================================================
class PrivateTestConfig:
    # Hãy dán đường dẫn file CSV Private Test của bạn vào đây:
    PRIVATE_CSV = "/kaggle/input/datasets/duckanh0/mdd-private-test/MDD-Challenge-2025-private-test/metadata/private_test_submission.csv"
    
    # Hãy dán đường dẫn thư mục chứa các file Audio (.wav) Private Test vào đây:
    PRIVATE_AUDIO_DIR = "/kaggle/input/datasets/duckanh0/mdd-private-test/MDD-Challenge-2025-private-test/audio_data/private_test"
    
    # Đường dẫn trọng số mô hình tốt nhất đã huấn luyện thành công ở Cell 7
    MODEL_WEIGHTS = "/kaggle/working/best_mdd_model.pt"

# Kiểm tra tính toàn vẹn của file cấu hình đường dẫn
if os.path.exists(PrivateTestConfig.PRIVATE_CSV):
    private_df = pd.read_csv(PrivateTestConfig.PRIVATE_CSV)
    print(f"✅ Đã kết nối trực tiếp đến tập PRIVATE TEST thành công!")
    print(f"📊 Tổng số lượng mẫu cần suy diễn: {len(private_df)} dòng.")
else:
    raise FileNotFoundError(f"❌ Lỗi: Không tìm thấy file CSV tại đường dẫn bạn điền: {PrivateTestConfig.PRIVATE_CSV}")

# ==========================================
# 2. KHỞI TẠO MODEL VÀ LOAD TRỌNG SỐ
# ==========================================
print("🔄 Đang nạp trọng số mô hình tốt nhất...")
final_model = MultitaskMDDModel(Config.MODEL_CHECKPOINT, len(char2idx), len(diag2idx))

if os.path.exists(PrivateTestConfig.MODEL_WEIGHTS):
    state_dict = torch.load(PrivateTestConfig.MODEL_WEIGHTS, map_location=device)
    
    # Gỡ bọc module. nếu mô hình được huấn luyện bằng Multi-GPU DataParallel
    first_key = next(iter(state_dict.keys()))
    if first_key.startswith("module."):
        from collections import OrderedDict
        new_state_dict = OrderedDict()
        for k, v in state_dict.items():
            new_state_dict[k[7:]] = v
        state_dict = new_state_dict
        
    final_model.load_state_dict(state_dict, strict=False)
    print(f"✅ Đã tải thành công trọng số: {PrivateTestConfig.MODEL_WEIGHTS}")
else:
    print(f"⚠️ Cảnh báo: Không tìm thấy file trọng số. Hệ thống sẽ dùng mạng ngẫu nhiên để test luồng.")

# Đưa mô hình về Single-GPU Eval Mode để xử lý độ dài âm thanh linh hoạt
if isinstance(final_model, nn.DataParallel):
    final_model = final_model.module
final_model = final_model.to(device).eval()

# Bảng tra cứu ID ngược ra Token âm vị
idx2char = {idx: token for token, idx in char2idx.items()}
char_pad_idx = char2idx["[PAD]"]

# ==========================================
# 3. THUẬT TOÁN CTC GREEDY DECODE 
# ==========================================
def ctc_greedy_decode_phoneme(pred_indices, idx2char, blank_idx):
    collapsed_phonemes = []
    previous_idx = None
    
    for idx in pred_indices:
        if idx != blank_idx:
            if idx != previous_idx:
                phoneme_token = idx2char.get(idx, "")
                if phoneme_token not in ["[UNK]", "[PAD]", "<unk>", "<eps>", ""]:
                    collapsed_phonemes.append(phoneme_token)
        previous_idx = idx
        
    return " ".join(collapsed_phonemes)

# ==========================================
# 4. SUY DIỄN FULL AUDIO (KHÔNG CHIA CHUNK)
# ==========================================
private_predict_results = []
missing_private_files = 0

print("🚀 Khởi chạy suy diễn Full Audio trên tập Private Test...")
with torch.no_grad():
    for idx in tqdm(range(len(private_df)), desc="Predicting Private Test"):
        row = private_df.iloc[idx]
        csv_path = str(row['path']).strip()
        
        # Đồng bộ hóa tên file audio thực tế (.wav)
        file_name = os.path.basename(csv_path)
        if not file_name.endswith('.wav'):
            file_name += '.wav'
            
        audio_path = os.path.join(PrivateTestConfig.PRIVATE_AUDIO_DIR, file_name)
        
        # Phương án dự phòng nếu đường dẫn trỏ thẳng từ file gốc
        if not os.path.exists(audio_path):
            alt_path = os.path.join(os.path.dirname(PrivateTestConfig.PRIVATE_CSV), csv_path)
            if not alt_path.endswith('.wav'): alt_path += '.wav'
            if os.path.exists(alt_path):
                audio_path = alt_path

        # Xử lý nếu thiếu file vật lý trên đĩa
        if not os.path.exists(audio_path):
            missing_private_files += 1
            private_predict_results.append("")
            continue
            
        # Đọc mảng sóng âm nguyên bản
        speech, sample_rate = sf.read(audio_path)
        
        # Ép về tần số chuẩn 16kHz nếu cần
        if sample_rate != 16000:
            import librosa
            speech = librosa.resample(speech, orig_sr=sample_rate, target_sr=16000)
            
        # Trích xuất đặc trưng acoustic từ toàn bộ file (Full Audio)
        inputs = processor(speech, sampling_rate=16000, return_tensors="pt", return_attention_mask=True)
        input_values = inputs.input_values.to(device)
        attention_mask = inputs.attention_mask.to(device)
        
        # Forward qua mạng Wav2Vec2 + MDD Head
        asr_logits, _ = final_model(input_values, attention_mask=attention_mask)
        full_audio_preds = asr_logits.argmax(dim=-1).squeeze(0).cpu().numpy()
        
        # Giải mã chuỗi âm vị liên tục
        predicted_phoneme_string = ctc_greedy_decode_phoneme(full_audio_preds, idx2char, char_pad_idx)
        private_predict_results.append(predicted_phoneme_string)
        
        # Dọn dẹp bộ nhớ định kỳ tránh tràn VRAM
        if idx % 50 == 0:
            torch.cuda.empty_cache()
            gc.collect()

if missing_private_files > 0:
    print(f"🛑 Chú ý: Phát hiện {missing_private_files}/{len(private_df)} file âm thanh không tìm thấy thực tế trong thư mục!")

# ==========================================
# 5. XUẤT THÀNH FILE SUBMISSION THEO FORMAT
# ==========================================
submission_df = pd.DataFrame({
    'id': private_df['id'],
    'path': private_df['path'],
    'predict': private_predict_results
})

# Khử khoảng trống thừa đầu và cuối chuỗi kết quả
submission_df['predict'] = submission_df['predict'].fillna("").astype(str).str.strip()

output_submission_path = "submission_private.csv"
submission_df.to_csv(output_submission_path, index=False, encoding='utf-8', lineterminator='\n')

print(f"\n✨ Quá trình xử lý Private Test kết thúc thành công!")
print(f"💾 File nộp bài đã được lưu tại: /kaggle/working/{output_submission_path}")
print(f"📊 Xem trước cấu trúc file nộp bài (3 cột chuẩn chỉ):")
print(submission_df.head(5))

✅ Đã kết nối trực tiếp đến tập PRIVATE TEST thành công!
📊 Tổng số lượng mẫu cần suy diễn: 856 dòng.
🔄 Đang nạp trọng số mô hình tốt nhất...


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: nguyenvulebinh/wav2vec2-base-vietnamese-250h
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 
lm_head.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Đã tải thành công trọng số: /kaggle/working/best_mdd_model.pt
🚀 Khởi chạy suy diễn Full Audio trên tập Private Test...


Predicting Private Test: 100%|██████████| 856/856 [05:48<00:00,  2.45it/s]


✨ Quá trình xử lý Private Test kết thúc thành công!
💾 File nộp bài đã được lưu tại: /kaggle/working/submission_private.csv
📊 Xem trước cấu trúc file nộp bài (3 cột chuẩn chỉ):
                        id                                               path  \
0  nang-giai_1618731750770  audio_data/private_test/nang-giai_161873175077...   
1  nang-giai_1618731781800  audio_data/private_test/nang-giai_161873178180...   
2  nang-giai_1618732605210  audio_data/private_test/nang-giai_161873260521...   
3  nang-giai_1618732684413  audio_data/private_test/nang-giai_161873268441...   
4  nang-giai_1618732695387  audio_data/private_test/nang-giai_161873269538...   

              predict  
0   n a-0 ŋz z a-1 iz  
1  n a-0 ŋz z aː-2 iz  
2   n a-0 ŋz z a-3 iz  
3  n a-0 ŋz z aː-4 iz  
4   n a-0 ŋz z a-3 iz  
